<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 1. Google Drive 연결 </h2>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 2. 데이터셋 ZIP 파일 경로 설정 </h2>

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/wardy/ml/src/data/person_dataset/person_data_merge/person_dataset_merged.zip"
LOCAL_ZIP = "/content/person_dataset_merged.zip"

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 3. ZIP 파일 Colab 로컬 환경으로 복사 </h2>

In [ ]:
import shutil

shutil.copy2(ZIP_PATH, LOCAL_ZIP)

print("ZIP 파일 복사 완료")

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 4. 데이터셋 ZIP 파일 압축 해제 </h2>

In [ ]:
import zipfile

with zipfile.ZipFile(LOCAL_ZIP, "r") as zip_ref:
    zip_ref.extractall("/content")

print("압축 해제 완료")

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 5. 데이터셋 경로 설정 </h2>

In [ ]:
DATASET_PATH = "/content/person_dataset"
DATA_YAML = f"{DATASET_PATH}/data.yaml"

print(DATASET_PATH)
print(DATA_YAML)

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 6. data.yaml 경로 수정 </h2>

In [ ]:
from pathlib import Path

yaml_path = Path(DATA_YAML)

text = yaml_path.read_text(encoding="utf-8")
text = text.replace("path: .", "path: /content/person_dataset")

yaml_path.write_text(text, encoding="utf-8")

print(yaml_path.read_text(encoding="utf-8"))

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 7. Ultralytics YOLO 설치 </h2>

In [ ]:
!pip install -q -U ultralytics

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 8. GPU 사용 가능 여부 확인 </h2>

In [ ]:
import torch

print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 9. 5 Epoch 시험 학습 </h2>

In [ ]:
from ultralytics import YOLO
from pathlib import Path

RUN_DIR = Path('/content/drive/MyDrive/wardy/ml/src/export/person_detector_smoke_test')
model = YOLO('yolo11n.pt')
results = model.train(
    data=DATA_YAML,
    epochs=5,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    project=str(RUN_DIR.parent),
    name=RUN_DIR.name,
    exist_ok=True,
    seed=42,
    plots=True,
)

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 10. 시험 학습 결과 확인 </h2>

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 1. 결과 폴더와 생성 파일 확인 </h3>

In [ ]:
from IPython.display import display, Image
print('결과 폴더:', RUN_DIR)
print('폴더 존재:', RUN_DIR.exists())
if RUN_DIR.exists():
    for item in sorted(RUN_DIR.iterdir()):
        print(item.name)

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 2. 학습 이미지와 정답 라벨 확인 </h3>

In [ ]:
for filename in ['train_batch0.jpg', 'train_batch1.jpg', 'train_batch2.jpg']:
    image_path = RUN_DIR / filename
    if image_path.exists():
        print(filename)
        display(Image(filename=str(image_path), width=1000))

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 3. 검증 정답과 모델 예측 비교 </h3>

In [ ]:
for filename in ['val_batch0_labels.jpg', 'val_batch0_pred.jpg']:
    image_path = RUN_DIR / filename
    if image_path.exists():
        print(filename)
        display(Image(filename=str(image_path), width=1000))

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 4. 학습 그래프 확인 </h3>

In [ ]:
results_path = RUN_DIR / 'results.png'
if results_path.exists():
    display(Image(filename=str(results_path), width=1100))
else:
    print('results.png을 찾을 수 없습니다:', results_path)

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 5. 혼동행렬 확인 </h3>

In [ ]:
for filename in ['confusion_matrix.png', 'confusion_matrix_normalized.png']:
    image_path = RUN_DIR / filename
    if image_path.exists():
        print(filename)
        display(Image(filename=str(image_path), width=900))

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 6. 성능 지표와 모델 파일 확인 </h3>

In [ ]:
BEST_MODEL = RUN_DIR / 'weights' / 'best.pt'
LAST_MODEL = RUN_DIR / 'weights' / 'last.pt'
print('best.pt:', BEST_MODEL.exists(), BEST_MODEL)
print('last.pt:', LAST_MODEL.exists(), LAST_MODEL)

if BEST_MODEL.exists():
    best_model = YOLO(str(BEST_MODEL))
    metrics = best_model.val(data=DATA_YAML, split='test', device=0, plots=True)
    print(f'Precision : {metrics.box.mp:.4f}')
    print(f'Recall    : {metrics.box.mr:.4f}')
    print(f'mAP50     : {metrics.box.map50:.4f}')
    print(f'mAP50-95  : {metrics.box.map:.4f}')

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 7. Test 이미지 사람 탐지 결과 확인 </h3>

In [ ]:
test_images = sorted((Path(DATASET_PATH) / 'images' / 'test').glob('*'))
if not test_images:
    raise FileNotFoundError('Test 이미지를 찾을 수 없습니다.')
prediction_results = best_model.predict(
    source=str(test_images[0]), conf=0.5, save=True, project=str(RUN_DIR), name='prediction', exist_ok=True
)
result_image = RUN_DIR / 'prediction' / test_images[0].name
print('테스트 이미지:', test_images[0])
if result_image.exists():
    display(Image(filename=str(result_image), width=1000))